In [1]:
import cv2
import numpy as np
import pyautogui
import os

from mediapipe.python.solutions import hands as mp_hands

# PyAutoGUI Settings
pyautogui.FAILSAFE = False # Prevents program from stopping if you hit a corner
screen_w, screen_h = pyautogui.size() # Gets your monitor resolution

We don't want the mouse to teleport; we want it to glide. We use a smoothing factor for this.

In [2]:
# Initialize Tracker
hand_tracker = mp_hands.Hands(
    max_num_hands=1, 
    min_detection_confidence=0.7, 
    min_tracking_confidence=0.7
)

# Parameters
frame_reduction = 100  # Creates a 'margin' so you don't have to reach the camera edges
smooth_factor = 2      # Higher = smoother but slower cursor
plocX, plocY = 0, 0    # Previous locations
clocX, clocY = 0, 0    # Current locations

print(f"Screen Resolution: {screen_w}x{screen_h}")

Screen Resolution: 1920x1080


# The Global Controller Loop

In [5]:
# --- Optimization Settings ---
smooth_factor = 2      
frame_reduction = 120  

# --- Logic State ---
is_pinched = False  # Tracks the physical state of the "mouse button"
plocX, plocY = 0, 0

cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success: break
    
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hand_tracker.process(rgb)
    
    if results.multi_hand_landmarks:
        lms = results.multi_hand_landmarks[0].landmark
        
        # Coordinates for Thumb (4) and Index (8)
        tx, ty = int(lms[4].x * w), int(lms[4].y * h)
        ix, iy = int(lms[8].x * w), int(lms[8].y * h)
        
        # 1. Map Thumb Coordinates to Screen
        mx = np.interp(tx, (frame_reduction, w - frame_reduction), (0, screen_w))
        my = np.interp(ty, (frame_reduction, h - frame_reduction), (0, screen_h))
        
        # 2. Smooth the Movement
        clocX = plocX + (mx - plocX) / smooth_factor
        clocY = plocY + (my - plocY) / smooth_factor
        
        # 3. Move Cursor
        pyautogui.moveTo(clocX, clocY, _pause=False)
        plocX, plocY = clocX, clocY
        
        # 4. Native Click & Drag Logic
        dist = np.hypot(ix - tx, iy - ty)
        
        if dist < 40: 
            # Fingers are touching
            if not is_pinched:
                # The moment they touch, press the mouse button DOWN
                pyautogui.mouseDown()
                is_pinched = True
            
            # Visual feedback: Green circle means "Button is held down"
            cv2.circle(frame, (tx, ty), 15, (0, 255, 0), cv2.FILLED)
            
        else:
            # Fingers are apart
            if is_pinched:
                # The moment they separate, release the mouse button UP
                pyautogui.mouseUp()
                is_pinched = False
                
            # Visual feedback: Purple circle means "Hovering / Moving"
            cv2.circle(frame, (tx, ty), 10, (255, 0, 255), cv2.FILLED)

    cv2.imshow("Ultimate Virtual Mouse", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

# Safety cleanup: ensure mouse button isn't stuck down if you quit while pinching
if is_pinched:
    pyautogui.mouseUp()
    
cap.release()
cv2.destroyAllWindows()

C:\Users\manim\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
